 ### Consolidate Yloader OHLCV Data (Windowed Batch Ingestion)



 This notebook reads all individual ticker `.csv` files generated by `Yloader`,

 filters them using a rolling temporal window ($T_{\max} - \Delta t \le \text{mtime}_i \le T_{\max}$)

 to seamlessly absorb weekend and holiday batch updates, combines them into a

 single multi-indexed DataFrame `(Ticker, Date)`, and exports to Parquet.

In [1]:
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Optional

import pandas as pd

# --- Configuration ---
# 1. Source directory containing Yloader CSV files
YLOADER_DATA_DIR = Path.home() / "Desktop" / "yloader"

# 2. Explicit destination path for consolidated Parquet storage
DESTINATION_PARQUET_PATH = Path(
    r"C:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet"
)
ROOT_DIR = DESTINATION_PARQUET_PATH.parent.parent
SRC_DIR = ROOT_DIR / "src"

for path in (ROOT_DIR, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.append(str(path))

# 3. CSV Schema specification (files without headers, leading column is 'Date')
CANONICAL_COLUMN_NAMES = ["Adj Open", "Adj High", "Adj Low", "Adj Close", "Volume"]

# 4. Ingestion Window Parameters (3 full days covers weekends + 3-day market holidays)
BATCH_LOOKBACK_DAYS = 3.0
BENCHMARK_ANCHOR_TICKER = "SPY"

print(f"Project Root Directory set to: {ROOT_DIR}")
print(f"Reading Yloader CSVs from:      {YLOADER_DATA_DIR}")
print(f"Output will be saved to:        {DESTINATION_PARQUET_PATH}")
print(f"Lookback Window (Delta t):      {BATCH_LOOKBACK_DAYS} days")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

Project Root Directory set to: C:\Users\ping\Files_win10\python\py311\stocks
Reading Yloader CSVs from:      C:\Users\ping\Desktop\yloader
Output will be saved to:        C:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
Lookback Window (Delta t):      3.0 days


 ### Step 1: Process and Combine CSV Data via Rolling Window Clustering



 Scans `YLOADER_DATA_DIR`, computes $T_{\max} = \max(\text{mtime})$, filters files falling within

 $[T_{\max} - \Delta t, T_{\max}]$, and concatenates into a chronologically sorted MultiIndex DataFrame.

In [2]:
def process_yloader_csvs(
    data_dir: Path,
    canonical_cols: List[str],
    lookback_days: float = 3.0,
    anchor_ticker: str = "SPY",
) -> Optional[pd.DataFrame]:
    """
    Reads and combines CSV files whose modification timestamp satisfies:
        T_max - Delta_t <= mtime_i <= T_max
    where T_max is the newest file mtime in data_dir, and Delta_t = lookback_days.

    Args:
        data_dir (Path): Directory containing Yloader CSV files.
        canonical_cols (List[str]): Expected OHLCV column names (excluding 'Date').
        lookback_days (float): Elapsed time window in days. Defaults to 3.0.
        anchor_ticker (str): Ticker to verify in the active cluster. Defaults to 'SPY'.

    Returns:
        Optional[pd.DataFrame]: MultiIndex DataFrame (Ticker, Date) or None if failed.
    """
    if not data_dir.is_dir():
        print(f"Error: Directory not found: {data_dir}")
        return None

    # Collect files and mtimes in a single traversal pass
    all_files = list(data_dir.glob("*.csv"))
    if not all_files:
        print(f"Error: No CSV files found inside {data_dir}")
        return None

    # Determine T_max and T_cutoff
    file_mtime_map = {f: f.stat().st_mtime for f in all_files}
    t_max_ts = max(file_mtime_map.values())
    t_max_dt = datetime.fromtimestamp(t_max_ts)

    delta_seconds = lookback_days * 86400.0
    t_cutoff_ts = t_max_ts - delta_seconds
    t_cutoff_dt = datetime.fromtimestamp(t_cutoff_ts)

    print("=" * 70)
    print(f"Directory Total CSV Files:       {len(all_files)}")
    print(f"Latest Modification (T_max):     {t_max_dt.strftime('%Y-%m-%d %H:%M:%S')}")
    print(
        f"Window Start (T_max - Delta_t):  {t_cutoff_dt.strftime('%Y-%m-%d %H:%M:%S')}"
    )
    print(
        f"Window Length:                   {lookback_days} days ({lookback_days * 24:.1f} hours)"
    )

    # Filter files within [T_max - Delta_t, T_max]
    csv_files = [
        f for f, mtime in file_mtime_map.items() if t_cutoff_ts <= mtime <= t_max_ts
    ]

    print(f"Files Matching Ingestion Window: {len(csv_files)} / {len(all_files)}")
    print("=" * 70)

    if not csv_files:
        print("Error: Zero files matched the temporal criteria.")
        return None

    # Sanity check: Ensure benchmark anchor is present
    anchor_filename = f"{anchor_ticker.upper()}.csv"
    matched_names = {f.name for f in csv_files}
    if anchor_filename not in matched_names:
        print(
            f"WARNING: Anchor ticker '{anchor_ticker}' was NOT found in the current cluster!"
        )
    else:
        print(f"Verified anchor ticker '{anchor_ticker}' present in batch cluster.")

    # Read and aggregate matching CSVs
    all_dataframes = []
    tickers_list = []
    expected_csv_cols = ["Date"] + canonical_cols

    for file_path in csv_files:
        ticker = file_path.stem
        try:
            df_temp = pd.read_csv(
                file_path,
                header=None,
                names=expected_csv_cols,
                parse_dates=["Date"],
                index_col="Date",
            )

            if df_temp.empty:
                print(f"Warning: File {file_path.name} is empty. Skipping.")
                continue

            all_dataframes.append(df_temp)
            tickers_list.append(ticker)

        except Exception as e:
            print(f"Error processing {file_path.name}: {e}. Skipping.")

    if not all_dataframes:
        print("Error: No data was successfully loaded. Aborting.")
        return None

    # Concatenate and sort index chronologically (Ticker ASC, Date ASC)
    multi_index_df = pd.concat(
        all_dataframes, keys=tickers_list, names=["Ticker", "Date"]
    )
    multi_index_df.sort_index(
        level=["Ticker", "Date"], ascending=[True, True], inplace=True
    )

    print("\nCSV processing complete. MultiIndex DataFrame successfully built.")
    return multi_index_df

 ### Execute Step 1: Ingestion & Sanity Checks

In [3]:
final_df = process_yloader_csvs(
    data_dir=YLOADER_DATA_DIR,
    canonical_cols=CANONICAL_COLUMN_NAMES,
    lookback_days=BATCH_LOOKBACK_DAYS,
    anchor_ticker=BENCHMARK_ANCHOR_TICKER,
)

if final_df is not None:
    print("\n--- Consolidated DataFrame Info ---")
    final_df.info()

    tickers = final_df.index.get_level_values("Ticker").unique()
    dates = final_df.index.get_level_values("Date").unique()
    print(f"\nUnique Tickers Loaded: {len(tickers)}")
    print(
        f"Date Span:            {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}"
    )

    print("\n--- Sample Tail Rows ---")
    display(final_df.tail())

Directory Total CSV Files:       1718
Latest Modification (T_max):     2026-10-06 05:06:13
Window Start (T_max - Delta_t):  2026-10-03 05:06:13
Window Length:                   3.0 days (72.0 hours)
Files Matching Ingestion Window: 1715 / 1718
Verified anchor ticker 'SPY' present in batch cluster.



CSV processing complete. MultiIndex DataFrame successfully built.

--- Consolidated DataFrame Info ---


<class 'pandas.core.frame.DataFrame'>
MultiIndex: 8590139 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-10-05 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 360.8+ MB



Unique Tickers Loaded: 1715
Date Span:            1997-01-02 to 2026-10-05

--- Sample Tail Rows ---


Adj Open  Adj High  Adj Low  Adj Close   Volume
Ticker Date                                                       
ZWS    2026-09-29     46.90     46.94    46.09      46.79   754300
       2026-09-30     46.40     47.11    46.40      46.47   871200
       2026-10-01     46.31     47.35    45.83      47.20   761200
       2026-10-02     47.88     48.56    47.40      47.67   841300
       2026-10-05     48.14     48.56    47.20      47.99  1181900

 ### Step 2: Save to Parquet

In [4]:
def save_dataframe_to_parquet(df: Optional[pd.DataFrame], dest_path: Path) -> None:
    """
    Saves a MultiIndex DataFrame to Parquet using pyarrow and zstd compression.
    """
    if df is None or df.empty:
        print("DataFrame is empty or None. Nothing to save.")
        return

    try:
        dest_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(dest_path, engine="pyarrow", compression="zstd", index=True)
        print(f"\nSuccessfully saved DataFrame to: {dest_path}")
    except Exception as e:
        print(f"\nError: Failed to save Parquet file. Details: {e}")


# Execute Step 2
save_dataframe_to_parquet(final_df, DESTINATION_PARQUET_PATH)


Successfully saved DataFrame to: C:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet


 ### Step 3: Verification Audit

In [5]:
print(f"--- Verifying Parquet Integrity at: {DESTINATION_PARQUET_PATH} ---")

try:
    if DESTINATION_PARQUET_PATH.exists():
        verified_df = pd.read_parquet(DESTINATION_PARQUET_PATH)
        print("Verification successful: File read cleanly from disk.")
        print(f"Shape: {verified_df.shape[0]:,} rows x {verified_df.shape[1]} columns")

        if not verified_df.empty:
            sample_ticker = (
                BENCHMARK_ANCHOR_TICKER
                if BENCHMARK_ANCHOR_TICKER in verified_df.index.levels[0]
                else verified_df.index.get_level_values("Ticker")[0]
            )
            print(f"\n--- Recent Bars for Anchor/Sample Ticker '{sample_ticker}' ---")
            display(verified_df.loc[sample_ticker].tail(5))
    else:
        print(f"Error: Target file not found at {DESTINATION_PARQUET_PATH}")

except Exception as e:
    print(f"Verification exception encountered: {e}")

--- Verifying Parquet Integrity at: C:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet ---


Verification successful: File read cleanly from disk.
Shape: 8,590,139 rows x 5 columns

--- Recent Bars for Anchor/Sample Ticker 'SPY' ---


,Adj Open,Adj High,Adj Low,Adj Close,Volume
Date,,,,,
2026-09-29,766.83,766.98,762.35,764.20,36910500
2026-09-30,766.45,769.41,762.18,762.63,62110000
2026-10-01,764.36,765.65,758.79,763.99,47708100
2026-10-02,770.58,772.65,767.15,769.64,46335300
2026-10-05,769.69,776.61,769.63,774.83,44789900


In [6]:
# final_df.loc["QQQ"]
final_df.loc["XLY"]

,Adj Open,Adj High,Adj Low,Adj Close,Volume
Date,,,,,
1998-12-22,9.28291,9.34021,9.28291,9.34021,10089
1998-12-23,9.35167,9.44336,9.34021,9.38033,32721
1998-12-24,9.51212,9.55223,9.46627,9.55223,4362
1998-12-28,9.50639,9.51212,9.46628,9.46628,23450
1998-12-29,9.49493,9.67257,9.49493,9.67257,2999
...,...,...,...,...,...
2026-09-29,109.35000,109.57000,108.68000,109.15000,7693300
2026-09-30,109.26000,109.70000,108.61000,108.84000,5635000
2026-10-01,109.21000,109.38000,107.99000,108.81000,7752600
